In [2]:
import pandas as pd
import numpy as np

In [11]:
PITCH_LENGTH  = 105.0
PITCH_WIDTH   = 68.0
GOAL_X        = 52.5          # goal line x in SkillCorner centred coords
GOAL_Y_TOP    =  3.66         # top post
GOAL_Y_BOT    = -3.66         # bottom post
BALL_ID       = 55            # SkillCorner open data — verify for your data
CONE_WIDTH_M  = 1.0           # lateral tolerance for defender-in-cone
PRESSURE_RADIUS_M = 3.0       # radius for pressure index


In [50]:
df = pd.read_csv('../data/full_data.csv')

/var/folders/rr/7hqln5dd7_55vxvrgjhmz1gw0000gn/T/ipykernel_3499/1643287228.py:1: DtypeWarning: Columns (0: game_interruption_after) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/full_data.csv')


In [ ]:
# Filter Data
df_copy = df.copy(deep=True)
df = df[df['end_type'] == 'shot']
shot_frames = df['frame_end'].unique()
df = df[df['frame'].isin(shot_frames)]

# Combine low occurance possession types into other ones
df.loc[df['team_in_possession_phase_type'] == 'direct', 'team_in_possession_phase_type'] = 'transition'
df.loc[df['team_in_possession_phase_type'] == 'build_up', 'team_in_possession_phase_type'] = 'transition'

In [14]:
coordinates = ['x', 'y', 'ball_x', 'ball_y']
df2 = df.copy(deep=True)
df2[coordinates] = df2[coordinates] * -1

df3 = df.copy(deep=True)
df3[['x', 'ball_x']] = df3[['x', 'ball_x']] * -1

df4 = df.copy(deep=True)
df4[['y', 'ball_y']] = df4[['y', 'ball_y']] * -1

In [15]:
# ── Constants ──────────────────────────────────────────────────────────────────
GOAL_X            =  52.5
GOAL_Y_TOP        =   3.66
GOAL_Y_BOT        =  -3.66
CONE_WIDTH_M      =   1.0
PRESSURE_RADIUS_M =   3.0


# ── Geometry ───────────────────────────────────────────────────────────────────
def normalise_direction(bx, by, frame_df, period):
    col = "direction_player_1st_half" if period == 1 else "direction_player_2nd_half"
    shooting_team_id  = frame_df["team_id_event"].iloc[0]
    shooter_direction = frame_df[frame_df["team_id"] == shooting_team_id][col].iloc[0]

    if shooter_direction == "right_to_left":
        bx = -bx
        by = -by
        frame_df = frame_df.copy()
        frame_df["x"] = -frame_df["x"]
        frame_df["y"] = -frame_df["y"]

    return bx, by, frame_df


def shot_distance(bx, by):
    return np.sqrt((bx - GOAL_X)**2 + by**2)


def shot_angle(bx, by):
    ax  = GOAL_X - bx;  ay  = GOAL_Y_TOP - by
    bvx = GOAL_X - bx;  bvy = GOAL_Y_BOT - by
    dot   = ax * bvx + ay * bvy
    cross = ax * bvy - ay * bvx
    return max(np.arctan2(abs(cross), dot), 0.0)


def defenders_in_cone(bx, by, defenders):
    if defenders.empty:
        return 0
    dx = GOAL_X - bx;  dy = 0.0 - by
    length = np.sqrt(dx**2 + dy**2)
    if length < 1e-6:
        return 0
    ux, uy = dx / length, dy / length
    px, py = -uy, ux
    rel_x  = defenders["x"].values - bx
    rel_y  = defenders["y"].values - by
    along   = rel_x * ux + rel_y * uy
    lateral = np.abs(rel_x * px + rel_y * py)
    return int(np.sum((along > 0) & (along < length) & (lateral < CONE_WIDTH_M)))


def gk_distance_from_line(defending_gk):
    if defending_gk.empty:
        return np.nan
    return abs(GOAL_X - defending_gk["x"].iloc[0])


def pressure_index(bx, by, defenders):
    if defenders.empty:
        return 0
    dists = np.sqrt((defenders["x"].values - bx)**2 +
                    (defenders["y"].values - by)**2)
    return int(np.sum(dists < PRESSURE_RADIUS_M))


# ── Feature extraction ─────────────────────────────────────────────────────────

def extract_shot_features(event_df):
    # Use frame closest to frame_end (the shot moment)
    frame_end        = event_df["frame_end"].iloc[0]
    available_frames = event_df["frame"].unique()
    best_frame       = min(available_frames, key=lambda f: abs(f - frame_end))
    frame_df         = event_df[event_df["frame"] == best_frame].copy()

    bx = frame_df["ball_x"].iloc[0]
    by = frame_df["ball_y"].iloc[0]

    shooting_team_id = frame_df["team_id_event"].iloc[0]
    period           = int(frame_df["period"].iloc[0])
    all_team_ids     = frame_df["team_id"].dropna().unique()
    defending_ids    = [t for t in all_team_ids if t != shooting_team_id]

    if not defending_ids:
        return None
    defending_team_id = defending_ids[0]

    bx, by, frame_df  = normalise_direction(bx, by, frame_df, period)

    defending_rows = frame_df[frame_df["team_id"] == defending_team_id]
    defending_gk   = defending_rows[defending_rows["is_gk"] == True]
    defenders_out  = defending_rows[defending_rows["is_gk"] == False]

    shooter_id  = frame_df["player_id_event"].iloc[0]
    shooter_row = frame_df[frame_df["player_id"] == shooter_id]

    return {
        "event_id":       frame_df["event_id"].iloc[0],
        "match_id":       frame_df["match_id"].iloc[0],
        "frame":          best_frame,
        "ball_x":         round(bx, 3),
        "ball_y":         round(by, 3),
        "distance":       round(shot_distance(bx, by), 3),
        "angle":          round(shot_angle(bx, by), 4),
        "pressure":       pressure_index(bx, by, defenders_out),
        "gk_dist":        round(gk_distance_from_line(defending_gk), 3),
        "defenders_cone": defenders_in_cone(bx, by, defenders_out),
        "one_touch":      int(frame_df["one_touch"].iloc[0]),
        "is_header":      int(frame_df["is_header"].iloc[0]),
        "phase_type":     str(frame_df["team_in_possession_phase_type"].iloc[0]),
        "goal":           int(frame_df["lead_to_goal"].iloc[0]),
    }


# ── Main ───────────────────────────────────────────────────────────────────────

def build_shot_dataset(df, output_path="shots_clean.csv"):
    shot_df = df[df["end_type"] == "shot"].copy()
    print(f"Found {shot_df['event_id'].nunique()} shot events across "
          f"{shot_df['match_id'].nunique()} matches")

    rows = []
    for (match_id, event_id), group in shot_df.groupby(["match_id", "event_id"]):
        features = extract_shot_features(group)
        if features is not None:
            rows.append(features)
        else:
            print(f"  Skipped match {match_id} event {event_id} — missing data")

    shots = pd.DataFrame(rows)

    print(f"\n=== Dataset summary ===")
    print(f"Total shots : {len(shots)}")
    print(f"Goals       : {shots['goal'].sum()}")
    print(f"Conversion  : {shots['goal'].mean():.1%}")
    print(f"\nphase_type counts (recoded):")
    print(shots["phase_type"].value_counts())
    print(f"\nMissing gk_dist: {shots['gk_dist'].isna().sum()}")

    shots = shots.dropna(subset=["gk_dist"])
    print(f"After dropping missing GK: {len(shots)} shots")

    shots.to_csv(output_path, index=False)
    print(f"\nSaved to {output_path}")

    return shots
# open_play = ['create', 'medium_block', 'high_block', 'low_block', 'build_up']



# ── Run ────────────────────────────────────────────────────────────────────────
build_shot_dataset(df, output_path="shots_clean.csv")
build_shot_dataset(df2, output_path="shots_clean2.csv")
build_shot_dataset(df3, output_path="shots_clean3.csv")
build_shot_dataset(df4, output_path="shots_clean4.csv")

Found 202 shot events across 10 matches

=== Dataset summary ===
Total shots : 226
Goals       : 26
Conversion  : 11.5%

phase_type counts (recoded):
phase_type
finish         118
set_play        57
transition      26
quick_break     25
Name: count, dtype: int64

Missing gk_dist: 0
After dropping missing GK: 226 shots

Saved to shots_clean.csv
Found 202 shot events across 10 matches

=== Dataset summary ===
Total shots : 226
Goals       : 26
Conversion  : 11.5%

phase_type counts (recoded):
phase_type
finish         118
set_play        57
transition      26
quick_break     25
Name: count, dtype: int64

Missing gk_dist: 0
After dropping missing GK: 226 shots

Saved to shots_clean2.csv
Found 202 shot events across 10 matches

=== Dataset summary ===
Total shots : 226
Goals       : 26
Conversion  : 11.5%

phase_type counts (recoded):
phase_type
finish         118
set_play        57
transition      26
quick_break     25
Name: count, dtype: int64

Missing gk_dist: 0
After dropping missing G

,event_id,match_id,frame,ball_x,ball_y,distance,angle,pressure,gk_dist,defenders_cone,one_touch,is_header,phase_type,goal
0,8_103,1886347,4236,28.82,1.06,23.704,0.3061,1,3.42,1,1,0,finish,0
1,8_131,1886347,5505,45.76,-13.89,15.439,0.2159,1,2.79,1,0,0,finish,0
2,8_244,1886347,10841,31.09,-12.85,24.970,0.2514,0,2.37,1,1,0,finish,0
3,8_261,1886347,12202,31.33,-16.13,26.615,0.2194,0,2.11,1,0,0,finish,0
4,8_276,1886347,12845,27.21,8.65,26.728,0.2582,0,2.79,1,0,0,finish,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,8_663,2017461,58416,38.86,16.51,21.416,0.2206,1,1.68,0,0,0,transition,0
222,8_683,2017461,59368,28.10,4.72,24.852,0.2874,1,2.83,2,0,0,quick_break,0
223,8_826,2017461,69501,37.08,6.75,16.833,0.3960,1,5.71,0,0,0,transition,0
224,8_838,2017461,69858,36.32,-12.12,20.216,0.2911,1,1.88,2,1,0,finish,0
